# Versioned SER Result Summary

Notebook gọn để tổng hợp tất cả `cross_session_summary.json` thành bảng mean ± std và heatmap theo held-out session. Notebook này không training, chỉ đọc kết quả đã có.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    display
except NameError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "results"
# Set to None or [] to read every results folder.
# Example: FOLDERS = ["versioned_loso", "cim_ablations", "ablation_model_loso"]
FOLDERS = ["versioned_loso", "cim_ablations", "cim_full_loso", "cim_ablation_loso", "fusion_model_loso", "cdm_ablation_loso"]
OUT_DIR = PROJECT_ROOT / "reports" / "versioned_result_summary"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["WA", "UA", "WF1", "Macro-F1"]
PRIMARY_METRIC = "Macro-F1"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR      =", OUT_DIR)
print("FOLDERS      =", FOLDERS if FOLDERS else "ALL")



## 2. Load Cross-Session Summaries


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def pct(value: float) -> float:
    return 100.0 * float(value)


def mean_std_text(mean: float, std: float) -> str:
    return f"{pct(mean):.2f} ± {pct(std):.2f}"


def infer_family(path: Path) -> str:
    parts = path.parts
    for key in [
        "versioned_loso",
        "ablation_model_loso",
        "ablation_loso",
        "cim_ablations",
        "cim_loso",
        "cim_full_loso",
        "cim_ablation_loso",
        "fusion_model_loso",
        "cdm_ablation_loso",
    ]:
        if key in parts:
            return key
    if "cross_session" in parts:
        return "legacy_loso"
    return "other"


def infer_model_name(path: Path, payload: dict) -> str:
    text = str(path)
    trainer = str(payload.get("trainer_module", ""))
    if "cim_zero" in text:
        return "CIM-zero"
    if "cim_shuffled" in text:
        return "CIM-shuffled"
    if "cim_no_" in text or "cim_overlap_only" in text:
        return path.parents[2].name if "cim_ablations" in path.parts else path.parents[3].name
    if "cim_ablation_loso" in text:
        if "no_relative_gap" in text:
            return "CIM ablation: no relative gap"
        if "no_overlap_ratio" in text:
            return "CIM ablation: no overlap ratio"
        if "no_speaker_switch" in text:
            return "CIM ablation: no speaker switch"
        if "no_speaker_overlap_style" in text:
            return "CIM ablation: no speaker overlap style"
        return path.parents[2].name
    if "cdm_ablation_loso" in text:
        if "zero_state" in text:
            return "CDM ablation: zero state"
        if "no_update" in text:
            return "CDM ablation: no update"
        if "shuffled_memory" in text:
            return "CDM ablation: shuffled memory"
        return path.parents[2].name
    if "fusion_model_loso" in text:
        if "residual_gated" in text:
            return "CIM fusion: residual gated"
        if "residual_sum" in text:
            return "CIM fusion: residual sum"
        if "branch_sum" in text:
            return "CIM fusion: branch sum"
        if "dialogue_only" in text:
            return "CIM fusion: dialogue only"
        if "temporal_only" in text:
            return "CIM fusion: interaction only"
        return path.parents[2].name
    if "cim_dual_interaction4" in text or "dual_interaction4" in text or "cim_full_loso" in text or "cim_loso" in text or "cim_branch_concat_interaction4" in text or "branch_concat_interaction4" in text:
        return "CIM dual interaction-4"
    if "cim_concat_interaction4" in text or "/concat_interaction4/" in text:
        return "CIM concat interaction-4"
    if "v3_2_cim_compact_primitives" in text:
        return "CIM v3.2 compact primitives"
    if "v3_1_cim_recommended_v2" in text or "wavlm_cim_v2_recommended" in text:
        return "CIM v3.1 recommended-v2"
    if "v2_2_2_dual_temporal_dialogue_fuse" in text or "wavlm_dual_branch_temporal_first" in text:
        return "Dual v2.2.2 temporal-first"
    if "v2_2_1_dual_dialogue_temporal_fuse" in text or "dual_branch_3phase" in text or "wavlm_dual_branch/cross_session" in text:
        return "Dual v2.2.1 3-phase"
    if "v2_1_dual_end2end" in text or "wavlm_dual_branch_end2end" in text:
        return "Dual v2.1 end-to-end"
    if "v1_cim_concat" in text or "wavlm_cim_end2end" in text or "wavlm_cim_loso" in text or trainer.endswith("train_wavlm_cim"):
        return "CIM v1 concat"
    if "wavlm_cdm" in text or trainer.endswith("train_wavlm_cdm"):
        return "CDM"
    if "wavlm_baseline" in text or trainer.endswith("train_wavlm_baseline"):
        return "Baseline"
    return path.parents[2].name if len(path.parents) > 2 else path.stem


def infer_version(path: Path, model_name: str) -> str:
    if model_name.startswith("CIM ablation"):
        return "CIM ablation"
    if model_name.startswith("CIM fusion"):
        return "CIM fusion"
    if model_name.startswith("CIM"):
        return "CIM"
    if model_name.startswith("CDM ablation"):
        return "CDM ablation"
    if "v3.1" in model_name:
        return "3.1"
    if "v2.2.2" in model_name:
        return "2.2.2"
    if "v2.2.1" in model_name:
        return "2.2.1"
    if "v2.1" in model_name:
        return "2.1"
    if "v1" in model_name:
        return "1"
    return "control"


def infer_setting(path: Path, payload: dict) -> str:
    cfg = payload.get("config", {}) if isinstance(payload.get("config"), dict) else {}
    text = str(path)
    if "setting_B" in text:
        return "B"
    if "setting_A" in text:
        return "A"
    # Existing results are frozen/precomputed fair runs unless config says otherwise.
    return "A"


def seed_from_path(path: Path) -> int | None:
    m = re.search(r"seed_(\d+)", str(path))
    return int(m.group(1)) if m else None


def result_top_folder(path: Path) -> str:
    try:
        rel = path.relative_to(RESULTS_DIR)
    except ValueError:
        return ""
    return rel.parts[0] if rel.parts else ""


def folder_allowed(path: Path) -> bool:
    if not FOLDERS:
        return True
    allowed = {str(folder).strip("/") for folder in FOLDERS}
    return result_top_folder(path) in allowed

all_summary_paths = sorted(RESULTS_DIR.glob("**/cross_session_summary.json"))
summary_paths = [path for path in all_summary_paths if folder_allowed(path)]
aggregate_rows = []
fold_rows = []

for path in summary_paths:
    payload = read_json(path)
    family = infer_family(path)
    model_name = infer_model_name(path, payload)
    version = infer_version(path, model_name)
    setting = infer_setting(path, payload)
    seed = seed_from_path(path)
    run_name = path.parent.name
    for metric in METRICS:
        item = payload.get("aggregate", {}).get(metric)
        if not item:
            continue
        aggregate_rows.append({
            "family": family,
            "setting": setting,
            "version": version,
            "model": model_name,
            "seed": seed,
            "run_name": run_name,
            "metric": metric,
            "mean": float(item["mean"]),
            "std": float(item["std"]),
            "n": int(item.get("n", 5)),
            "summary_path": str(path),
        })
    for fold in payload.get("folds", []):
        for metric in METRICS:
            if metric not in fold.get("metrics", {}):
                continue
            fold_rows.append({
                "family": family,
                "setting": setting,
                "version": version,
                "model": model_name,
                "seed": seed,
                "run_name": run_name,
                "test_session": int(fold["test_session"]),
                "metric": metric,
                "value": float(fold["metrics"][metric]),
                "summary_path": str(path),
            })

aggregate = pd.DataFrame(aggregate_rows)
folds = pd.DataFrame(fold_rows)
aggregate.to_csv(OUT_DIR / "all_cross_session_aggregate_long.csv", index=False)
folds.to_csv(OUT_DIR / "all_cross_session_folds_long.csv", index=False)
print("summaries found:", len(summary_paths), "/", len(all_summary_paths))
print("folders included:", sorted({result_top_folder(path) for path in summary_paths}))
print("aggregate rows:", len(aggregate))
display(aggregate.head())



## Prediction File Index

Log đường dẫn tới `predictions.csv` của từng version/run/fold để dễ đối chiếu demo, paired error và qualitative analysis.


In [ ]:
def row_count_csv(path: Path) -> int | None:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        return max(0, sum(1 for _ in handle) - 1)


def resolve_prediction_candidates(summary_path: Path, fold: dict) -> list[Path]:
    test_session = int(fold["test_session"])
    candidates: list[Path] = []

    output_dir = fold.get("output_dir")
    if output_dir:
        out = Path(output_dir)
        if not out.is_absolute():
            out = PROJECT_ROOT / out
        candidates.append(out / "predictions.csv")

        # Some historical summaries were generated before moving/copying result folders.
        output_text = str(output_dir)
        for old, new in [
            ("fair_ablation_loso", "ablation_model_loso"),
            ("ablation_loso", "ablation_model_loso"),
        ]:
            if old in output_text:
                candidates.append(PROJECT_ROOT / output_text.replace(old, new) / "predictions.csv")

    candidates.append(summary_path.parent / f"test_Ses{test_session:02d}" / "predictions.csv")
    candidates.append(summary_path.parent / f"test_Ses{test_session}" / "predictions.csv")

    deduped = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate not in seen:
            deduped.append(candidate)
            seen.add(candidate)
    return deduped


def first_existing_path(paths: list[Path]) -> Path:
    return next((p for p in paths if p.exists()), paths[0])


prediction_rows = []
for path in summary_paths:
    payload = read_json(path)
    family = infer_family(path)
    model_name = infer_model_name(path, payload)
    version = infer_version(path, model_name)
    setting = infer_setting(path, payload)
    seed = seed_from_path(path)
    run_name = path.parent.name

    for fold in payload.get("folds", []):
        test_session = int(fold["test_session"])
        prediction_path = first_existing_path(resolve_prediction_candidates(path, fold))
        fold_dir = prediction_path.parent
        exists = prediction_path.exists()
        prediction_rows.append({
            "family": family,
            "setting": setting,
            "version": version,
            "model": model_name,
            "seed": seed,
            "run_name": run_name,
            "test_session": test_session,
            "stage": "final",
            "exists": exists,
            "n_rows": row_count_csv(prediction_path),
            "prediction_path": str(prediction_path),
            "prediction_path_relative": str(prediction_path.relative_to(PROJECT_ROOT)) if exists and prediction_path.is_relative_to(PROJECT_ROOT) else str(prediction_path),
            "summary_path": str(path),
        })

        for phase_path in sorted(fold_dir.glob("phase_tests/*/predictions.csv")) if fold_dir.exists() else []:
            prediction_rows.append({
                "family": family,
                "setting": setting,
                "version": version,
                "model": model_name,
                "seed": seed,
                "run_name": run_name,
                "test_session": test_session,
                "stage": phase_path.parent.name,
                "exists": True,
                "n_rows": row_count_csv(phase_path),
                "prediction_path": str(phase_path),
                "prediction_path_relative": str(phase_path.relative_to(PROJECT_ROOT)) if phase_path.is_relative_to(PROJECT_ROOT) else str(phase_path),
                "summary_path": str(path),
            })

prediction_index = pd.DataFrame(prediction_rows)
prediction_index.to_csv(OUT_DIR / "prediction_file_index.csv", index=False)

# Historical result folders sometimes contain copied summaries that point to the same prediction files.
prediction_index_unique = (
    prediction_index
    .sort_values(["setting", "version", "model", "run_name", "test_session", "stage", "family"])
    .drop_duplicates(["prediction_path_relative"], keep="first")
    .reset_index(drop=True)
)
prediction_index_unique.to_csv(OUT_DIR / "prediction_file_index_unique.csv", index=False)

missing_predictions = prediction_index[~prediction_index["exists"]].copy()
missing_predictions.to_csv(OUT_DIR / "missing_prediction_files.csv", index=False)

print("prediction file entries:", len(prediction_index))
print("unique prediction paths:", len(prediction_index_unique))
print("existing unique prediction paths:", int(prediction_index_unique["exists"].sum()))
print("missing prediction file entries:", len(missing_predictions))
display(
    prediction_index_unique.sort_values(["setting", "version", "model", "run_name", "test_session", "stage"])[
        ["setting", "version", "model", "run_name", "test_session", "stage", "exists", "n_rows", "prediction_path_relative"]
    ].head(100)
)




## 3. Main Model Table


In [ ]:
MAIN_MODEL_ORDER = [
    "Baseline",
    "CDM",
    "CIM v1 concat",
    "CIM v3.1 recommended-v2",
    "CIM v3.2 compact primitives",
    "CIM concat interaction-4",
    "CIM dual interaction-4",
]
MAIN_FAMILIES = [
    "versioned_loso",
    "cim_loso",
    "cim_full_loso",
    "ablation_model_loso",
    "ablation_loso",
    "legacy_loso",
]

main = aggregate[
    aggregate["family"].isin(MAIN_FAMILIES)
    & aggregate["model"].isin(MAIN_MODEL_ORDER)
].copy()
family_priority = {
    "versioned_loso": 0,
    "cim_loso": 0,
    "cim_full_loso": 0,
    "ablation_model_loso": 1,
    "ablation_loso": 2,
    "legacy_loso": 3,
}
main["family_priority"] = main["family"].map(family_priority).fillna(9)
main = main.sort_values(["model", "metric", "family_priority", "run_name"], ascending=[True, True, True, False]).drop_duplicates(["setting", "model", "metric"], keep="first")

main_table = (
    main.assign(mean_std=lambda df: [mean_std_text(m, s) for m, s in zip(df["mean"], df["std"])])
    .pivot_table(index=["setting", "version", "model"], columns="metric", values="mean_std", aggfunc="first")
)
main_table = main_table.reindex(columns=METRICS)
ordered_index = []
for setting in sorted(main["setting"].unique()):
    for model in MAIN_MODEL_ORDER:
        matches = [idx for idx in main_table.index if idx[0] == setting and idx[2] == model]
        ordered_index.extend(matches)
main_table = main_table.loc[ordered_index] if ordered_index else main_table
main_table.to_csv(OUT_DIR / "main_model_summary_table.csv")
display(main_table)


## 4. Main Model Heatmaps


In [ ]:
for metric in ["Macro-F1", "UA", "WA"]:
    metric_df = main[main["metric"].eq(metric)].copy()
    if metric_df.empty:
        continue
    heat = metric_df.pivot_table(index="model", columns="setting", values="mean", aggfunc="max") * 100.0
    heat = heat.reindex([m for m in MAIN_MODEL_ORDER if m in heat.index])
    heat.to_csv(OUT_DIR / f"main_model_{metric.lower().replace('-', '_')}_by_setting.csv")
    fig, ax = plt.subplots(figsize=(7, max(3, 0.45 * len(heat))))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"Main models {metric} mean (%)")
    ax.set_xlabel("Setting")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"main_model_{metric.lower().replace('-', '_')}_heatmap.png", dpi=180)
    plt.show()


## 5. Held-Out Session Heatmap


In [ ]:
main_folds = folds[
    folds["model"].isin(MAIN_MODEL_ORDER)
    & folds["metric"].eq(PRIMARY_METRIC)
    & folds["family"].isin(MAIN_FAMILIES)
].copy()
main_folds["family_priority"] = main_folds["family"].map(family_priority).fillna(9)
main_folds = main_folds.sort_values(["model", "test_session", "family_priority", "run_name"], ascending=[True, True, True, False]).drop_duplicates(["setting", "model", "test_session"], keep="first")

for setting, frame in main_folds.groupby("setting"):
    pivot = frame.pivot_table(index="model", columns="test_session", values="value", aggfunc="mean") * 100.0
    pivot = pivot.reindex([m for m in MAIN_MODEL_ORDER if m in pivot.index])
    pivot.to_csv(OUT_DIR / f"main_model_{PRIMARY_METRIC.lower().replace('-', '_')}_by_session_setting_{setting}.csv")
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"Main models {PRIMARY_METRIC} by held-out session - Setting {setting} (%)")
    ax.set_xlabel("Held-out session")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"main_model_{PRIMARY_METRIC.lower().replace('-', '_')}_by_session_setting_{setting}.png", dpi=180)
    plt.show()

display(main_folds.head())


## Ablation And Fusion Tables

CIM ablations, CIM feature ablations, CIM fusion variants, and CDM memory ablations are reported separately from the main model table.


In [ ]:
def make_group_table(frame: pd.DataFrame, name: str, order: list[str] | None = None):
    if frame.empty:
        print(f"No {name} summaries found.")
        return pd.DataFrame()
    table = (
        frame.assign(mean_std=lambda df: [mean_std_text(m, s) for m, s in zip(df["mean"], df["std"])])
        .pivot_table(index="model", columns="metric", values="mean_std", aggfunc="first")
        .reindex(columns=METRICS)
    )
    if order:
        ordered = [model for model in order if model in table.index]
        ordered.extend(sorted(model for model in table.index if model not in ordered))
        table = table.reindex(ordered)
    table.to_csv(OUT_DIR / f"{name}_summary_table.csv")
    display(table)

    heat = frame[frame["metric"].eq(PRIMARY_METRIC)].pivot_table(index="model", columns="setting", values="mean", aggfunc="max") * 100.0
    if order:
        ordered = [model for model in order if model in heat.index]
        ordered.extend(sorted(model for model in heat.index if model not in ordered))
        heat = heat.reindex(ordered)
    heat.to_csv(OUT_DIR / f"{name}_{PRIMARY_METRIC.lower().replace('-', '_')}_heatmap.csv")
    fig, ax = plt.subplots(figsize=(6, max(3, 0.45 * len(heat))))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"{name.replace('_', ' ').title()} {PRIMARY_METRIC} (%)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}_{PRIMARY_METRIC.lower().replace('-', '_')}_heatmap.png", dpi=180)
    plt.show()
    return table

CIM_ABLATION_ORDER = [
    "CIM-zero",
    "CIM-shuffled",
    "cim_no_duration",
    "cim_no_gap",
    "cim_no_overlap",
    "cim_no_speaker_switch",
    "cim_no_turn_position",
    "cim_overlap_only",
]
CIM_FUSION_ORDER = [
    "CIM fusion: residual gated",
    "CIM fusion: residual sum",
    "CIM fusion: branch sum",
    "CIM fusion: dialogue only",
    "CIM fusion: interaction only",
]
CIM_FEATURE_ABLATION_ORDER = [
    "CIM ablation: no relative gap",
    "CIM ablation: no overlap ratio",
    "CIM ablation: no speaker switch",
    "CIM ablation: no speaker overlap style",
]
CDM_MEMORY_ABLATION_ORDER = [
    "CDM ablation: zero state",
    "CDM ablation: no update",
    "CDM ablation: shuffled memory",
]

cim_ablation = aggregate[aggregate["family"].eq("cim_ablations")].copy()
cim_fusion = aggregate[aggregate["family"].eq("fusion_model_loso")].copy()
cim_feature_ablation = aggregate[aggregate["family"].eq("cim_ablation_loso")].copy()
cdm_memory_ablation = aggregate[aggregate["family"].eq("cdm_ablation_loso")].copy()

cim_ablation_table = make_group_table(cim_ablation, "cim_ablation", CIM_ABLATION_ORDER)
cim_fusion_table = make_group_table(cim_fusion, "cim_fusion", CIM_FUSION_ORDER)
cim_feature_ablation_table = make_group_table(cim_feature_ablation, "cim_feature_ablation", CIM_FEATURE_ABLATION_ORDER)
cdm_memory_ablation_table = make_group_table(cdm_memory_ablation, "cdm_memory_ablation", CDM_MEMORY_ABLATION_ORDER)


## 7. Markdown Report


In [ ]:
report = f"""
# Versioned Result Summary

## Main model table

{main_table.to_markdown() if not main_table.empty else 'No main model results found.'}

## CIM ablation table

{ablation_table.to_markdown() if 'ablation_table' in globals() and not ablation_table.empty else 'No CIM ablation results found.'}

## Generated files

- `all_cross_session_aggregate_long.csv`
- `all_cross_session_folds_long.csv`
- `prediction_file_index.csv`
- `prediction_file_index_unique.csv`
- `missing_prediction_files.csv`
- `main_model_summary_table.csv`
- `main_model_macro_f1_by_setting.csv`
- `main_model_macro_f1_by_session_setting_A.csv`
- `cim_ablation_summary_table.csv`
- figures under `figures/`
"""
(OUT_DIR / "versioned_result_summary.md").write_text(report.strip() + "\n", encoding="utf-8")
print(report)




## 8. Saved Outputs


In [ ]:
outputs = sorted(p.relative_to(OUT_DIR) for p in OUT_DIR.rglob("*") if p.is_file())
print(f"Saved {len(outputs)} files under {OUT_DIR}")
for path in outputs:
    print(path)
